# ASHE earnings (residence-based) → LAD-level table, 2002–2025

Pulls **ASHE resident-analysis earnings at local-authority level** from the Nomis API and saves a tidy table: `clean/ashe_residence_lad.csv`.

- **Source:** Nomis **NM_30_1** (ASHE, resident analysis), **full-time workers** (`sex=8`). Per LA per year: **number of jobs**, **median**, **percentiles (10–90)**, for **gross weekly** (`pay=1`) and **gross hourly** (`pay=5`) pay, each with its **CV** (Nomis calls the measure *"Confidence"*; it is the coefficient of variation, %). GB only — **NI is not in NM_30_1**.
- **Kept at LA level on purpose.** Medians/percentiles can't be summed or averaged across areas, so nothing is aggregated here. Each row is an LA unit *as published*, tagged with its **current LA** (`la_code_current`, via `lib/la_code_crosswalk.csv`) and **ITL2** (via `clean/geography_spine.csv`). Jobs + percentiles are the ingredients for the later jobs-weighted percentile interpolation to ITL2.

**Why several geography vintages.** Nomis files each year's estimates under the LA codes valid in a given boundary set, and no single set holds the whole history:

| Nomis geography | holds |
|---|---|
| `TYPE424` LADs (April 2023) | 2008–2025, but reorganised LAs only from when they existed in that set |
| `TYPE432` / `TYPE434` LADs (April 2021 / 2019) | e.g. East/West Suffolk 2018–21, North/West Northamptonshire 2020–21 |
| `TYPE464` LADs (pre-April 2015) | 2002–2007 |
| **predecessor codes** (every old code in the crosswalk) | pre-merger districts, e.g. Cornwall's 6 districts pre-2008, Allerdale/Carlisle… pre-2022, Bournemouth/Poole pre-2018 |

**Stitching rule (§2b):**
1. A given LA code + year can appear in several vintages. They almost always agree; where they don't, the **typed vintage pulls are trusted over the code-queried predecessor pull** and the newest vintage with data wins. (Known case: querying St Edmundsbury's code directly returns Forest Heath's figures for 2008–17; the pre-2015 vintage has the right ones.)
2. For each **current LA × year**: use the current LA itself if it has data; otherwise use its **predecessors** (via the crosswalk). Never both → **no area counted twice** (checked). Every unit nests in one ITL2 because reorganisations merged whole districts.
3. Predecessor units that **duplicate each other** — identical values, or the same area under successive codes (recodes) — are kept **once**, otherwise jobs would be double-weighted later.

**Known data limits (checked in §3):**
- **Suppressed cells are kept as NaN** with their Nomis `status` (`B` = CV > 20%, unreliable; `F` = sample 0–2; `Q` = missing). So CV never exceeds 20 — use the ONS bands in `cv_band`: ≤5 precise, 5–10 reasonably precise, 10–20 use with caution.
- **P90 is suppressed for ~90% of LAs, P80 for ~15%** — the upper tail is effectively unavailable at LA level.
- **Number of jobs is rounded to the nearest 1,000.**
- **ASHE series breaks:** 2004 (methodology change; 2002–03 are on the earlier basis), 2006 (questionnaire change), 2011 (SOC 2010), 2021 (SOC 2020). Treat level shifts across these with care.
- **Isles of Scilly** and **City of London** are missing in most years (tiny samples) under every vintage.
- **⚠ Must page the API.** The Nomis API caps each request at 25,000 rows and truncates silently, so every fetch pages and is checked for completeness.

In [1]:
import time
from datetime import date
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
SPINE     = BASE / "clean" / "geography_spine.csv"
RAW_DIR   = BASE / "raw" / "ashe"
OUT       = BASE / "clean" / "ashe_residence_lad.csv"

# Nomis LA geography vintages, newest first (= priority when the same code-year is in several)
VINTAGES = {"TYPE424": "LAD 2023", "TYPE432": "LAD 2021", "TYPE434": "LAD 2019", "TYPE464": "LAD pre-2015"}

QUERY = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_30_1.data.csv"
         "?geography={geo}"
         "&sex=8"                      # full-time workers
         "&item=1,2,6...15"            # jobs, median, P10-P90
         "&pay=1,5"                    # gross weekly, gross hourly
         "&measures=20100,20701"       # value, confidence (= CV %)
         "&select=date_name,geography_code,geography_name,pay_name,item_name,"
         "measures_name,obs_value,obs_status")

xw    = pd.read_csv(CROSSWALK, dtype=str)
remap = dict(zip(xw.old_code, xw.new_code))
spine = pd.read_csv(SPINE)
lad2itl = dict(zip(spine.la_code, spine.itl2_code))
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Fetch from Nomis, paging past the 25,000-row cap

Requests 25,000 rows at a time until a short page signals the end, retrying a page if Nomis drops the connection. Each extract is saved, dated, to `raw/ashe/` and checked to be a complete, duplicate-free grid. Five pulls: the four LA vintages plus every crosswalk predecessor code (~2–3 min in total).

In [2]:
def read_page(url, tries=3):
    for k in range(tries):
        try:
            return pd.read_csv(url, encoding="utf-8-sig")
        except Exception as e:
            if k == tries - 1: raise
            print(f"  retry ({e})"); time.sleep(5)

KEYS = ["DATE_NAME","GEOGRAPHY_CODE","PAY_NAME","ITEM_NAME","MEASURES_NAME"]

def fetch_nomis_all(url, tag, limit=25000):
    frames, offset = [], 0
    while True:
        part = read_page(f"{url}&recordlimit={limit}&recordoffset={offset}")
        frames.append(part)
        if len(part) < limit:
            break
        offset += limit
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    # completeness: every date x geography x pay x item x measure exactly once
    grid = 1
    for k in KEYS: grid *= df[k].nunique()
    assert not df.duplicated(KEYS).any(), f"{tag}: duplicate rows - paging overlapped"
    assert len(df) == grid, f"{tag}: {len(df):,} rows but full grid is {grid:,} - paging lost rows"
    raw_path = RAW_DIR / f"nomis_ashe_{tag}_{date.today().isoformat()}.csv"
    df.to_csv(raw_path, index=False)
    print(f"{tag}: {len(df):,} rows, {df.GEOGRAPHY_CODE.nunique()} areas, "
          f"{df.DATE_NAME.nunique()} years -> {raw_path.name}")
    return df

raws = {VINTAGES[t]: fetch_nomis_all(QUERY.format(geo=t), f"residence_{t}") for t in VINTAGES}
raws["predecessor code"] = fetch_nomis_all(QUERY.format(geo=",".join(sorted(set(xw.old_code)))),
                                           "residence_predecessors")

residence_TYPE424: 403,200 rows, 350 areas, 24 years -> nomis_ashe_residence_TYPE424_2026-10-07.csv


residence_TYPE432: 418,176 rows, 363 areas, 24 years -> nomis_ashe_residence_TYPE432_2026-10-07.csv


residence_TYPE434: 427,392 rows, 371 areas, 24 years -> nomis_ashe_residence_TYPE434_2026-10-07.csv


residence_TYPE464: 437,760 rows, 380 areas, 24 years -> nomis_ashe_residence_TYPE464_2026-10-07.csv


residence_predecessors: 109,440 rows, 95 areas, 24 years -> nomis_ashe_residence_predecessors_2026-10-07.csv


## 2. Tidy to an LA-level table

One row per LA unit (as published) × year × pay type × item, with `value`, `cv` and Nomis `status`. **Suppressed cells stay in as NaN** (a `pivot_table` would silently drop them). Each row carries `la_code_current` (crosswalk) and `itl2_code` (spine), plus `geo_vintage` saying which pull it came from.

In [3]:
idx = ["la_code","la_name","year","pay_type","item"]

def tidy(raw, vintage):
    df = raw.rename(columns={"GEOGRAPHY_CODE":"la_code","GEOGRAPHY_NAME":"la_name","PAY_NAME":"pay_type",
                             "ITEM_NAME":"item","OBS_VALUE":"value","OBS_STATUS":"status","MEASURES_NAME":"measure"})
    df["year"]     = df["DATE_NAME"].astype(str).str.extract(r"(\d{4})")[0].astype(int)
    df["la_code"]  = df["la_code"].astype(str).str.strip()
    df["la_name"]  = df["la_name"].str.replace(r"\s*\(.*\)$", "", regex=True).str.strip()   # drop "(Apr 2019 merged ...)"
    df["value"]    = pd.to_numeric(df["value"], errors="coerce")
    df["pay_type"] = df["pay_type"].str.replace(" pay - gross","",regex=False).str.strip()
    df["measure"]  = df["measure"].str.strip().str.lower().map({"value":"value", "confidence":"cv"})
    assert df["measure"].notna().all(), "unexpected measure name from Nomis"
    out = df.set_index(idx + ["measure"])["value"].unstack("measure").reset_index()   # keeps NaN cells
    out = out.join(df[df.measure == "value"].set_index(idx)["status"], on=idx)
    out.columns.name = None
    out["la_code_current"] = out["la_code"].replace(remap)
    out["itl2_code"]       = out["la_code_current"].map(lad2itl)
    out["geo_vintage"]     = vintage
    return out

tidied = {v: tidy(r, v) for v, r in raws.items()}
for v, t in tidied.items():
    bad = sorted(t.loc[t.itl2_code.isna(), "la_code"].unique())
    print(f"{v:17s}: {t.la_code.nunique()} codes | no ITL2 via crosswalk+spine: {bad or 'none'}")

LAD 2023         : 350 codes | no ITL2 via crosswalk+spine: none
LAD 2021         : 363 codes | no ITL2 via crosswalk+spine: none
LAD 2019         : 371 codes | no ITL2 via crosswalk+spine: none
LAD pre-2015     : 380 codes | no ITL2 via crosswalk+spine: none
predecessor code : 95 codes | no ITL2 via crosswalk+spine: none


### 2b. Stitch the vintages

1. **Same code, several vintages** → report any disagreement, keep the newest typed vintage that has data for that code-year (the code-queried predecessor pull ranks last).
2. **Per current LA × year** → the current LA itself if it has data, else its predecessors (never both).
3. **Predecessors duplicating each other** (identical published values, or the same area under successive codes) → kept once.

Checks: no double coverage, no duplicate rows.

In [4]:
key  = ["la_code","year","pay_type","item"]
prio = {v: i for i, v in enumerate(tidied)}                 # 0 = newest vintage
allv = pd.concat(tidied.values(), ignore_index=True)
allv["prio"] = allv.geo_vintage.map(prio)

# (1) same code-year in several vintages: values must agree; keep newest vintage with data
both = allv.dropna(subset=["value"]).groupby(key).value.agg(["min","max"])
dis = both[both["max"] > both["min"]].reset_index()
print(f"Cells where vintages disagree for the same code-year: {len(dis)}  (resolved by vintage priority)")
if len(dis):
    print(dis.groupby("la_code").year.agg(lambda s: f"{s.min()}-{s.max()}").to_string())
unit_has = allv.groupby(["la_code","year","geo_vintage","prio"]).value.count().reset_index(name="n")
best = (unit_has[unit_has.n > 0].sort_values("prio").drop_duplicates(["la_code","year"])
        [["la_code","year","geo_vintage"]])
units = allv.merge(best, on=["la_code","year","geo_vintage"])

# (2) per current LA-year: itself if it has data, else predecessors
self_years = set(units.loc[units.la_code == units.la_code_current, ["la_code","year"]].itertuples(index=False, name=None))
units = units[(units.la_code == units.la_code_current) |
              pd.Series([ (c, y) not in self_years for c, y in zip(units.la_code_current, units.year) ], index=units.index)]

# (3) predecessors that duplicate each other: identical values, or same area under successive codes
vec = (units.set_index(["la_code_current","year","la_code","la_name","prio","pay_type","item"])["value"]
            .unstack(["pay_type","item"]))
vec = vec.round(2).astype(str).agg("|".join, axis=1).rename("sig").reset_index()
vec = vec.sort_values(["prio","la_code"])
dupe = vec.duplicated(["la_code_current","year","sig"]) | vec.duplicated(["la_code_current","year","la_name"])
dropped = vec.loc[dupe, ["la_code_current","year","la_code","la_name"]]
print(f"Duplicate predecessor units dropped: {len(dropped)}")
print(dropped.groupby(["la_code_current","la_code","la_name"]).year
             .agg(lambda s: f"{s.min()}-{s.max()} ({len(s)} yrs)").to_string())
drop_keys = set(dropped[["la_code","year"]].itertuples(index=False, name=None))
ashe = units[[ (l, y) not in drop_keys for l, y in zip(units.la_code, units.year) ]].drop(columns="prio")

# report gap-fills
pred_used = ashe[(ashe.la_code != ashe.la_code_current)].dropna(subset=["value"])
print("\nCurrent LAs filled from predecessors:")
print(pred_used.groupby("la_code_current").agg(units=("la_code","nunique"),
      years=("year", lambda s: f"{s.min()}-{s.max()}")).to_string())

# checks: no double coverage, no duplicate rows
has = ashe.dropna(subset=["value"])
lvl = has.assign(is_self=has.la_code == has.la_code_current).groupby(["la_code_current","year"]).is_self.nunique()
assert (lvl == 1).all(), f"double coverage: {lvl[lvl > 1].index.tolist()[:5]}"
assert not ashe.duplicated(key).any(), "duplicate LA-year-pay-item rows"

ashe["cv_band"] = pd.cut(ashe["cv"], [0, 5, 10, 20], labels=["precise","reasonably precise","use with caution"])
cols = ["la_code","la_name","la_code_current","itl2_code","geo_vintage","year","pay_type","item",
        "value","cv","cv_band","status"]
ashe = ashe[cols].sort_values(["year","itl2_code","la_code","pay_type","item"]).reset_index(drop=True)
ashe = ashe[ashe.year.isin(ashe.dropna(subset=["value"]).year.unique())]

OUT.parent.mkdir(parents=True, exist_ok=True)
ashe.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}  ({len(ashe):,} rows, {ashe.year.min()}-{ashe.year.max()})")
ashe.head(12)

Cells where vintages disagree for the same code-year: 198  (resolved by vintage priority)
la_code
E07000204    2008-2017


Duplicate predecessor units dropped: 0
Series([], )

Current LAs filled from predecessors:
                 units      years
la_code_current                  
E06000049            3  2002-2007
E06000050            3  2002-2007
E06000052            6  2002-2007
E06000056            2  2002-2007
E06000058            3  2002-2017
E06000059            5  2002-2007
E06000060            4  2002-2007
E06000061            4  2002-2019
E06000062            3  2002-2019
E06000063            3  2002-2021
E06000064            3  2002-2021
E06000065            7  2002-2007
E06000066            5  2002-2007
E07000244            2  2002-2017
E07000245            2  2002-2017
E08000038            1  2002-2025
E08000039            1  2002-2025



Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/ashe_residence_lad.csv  (210,432 rows, 2002-2025)


,la_code,la_name,la_code_current,itl2_code,geo_vintage,year,pay_type,item,value,cv,cv_band,status
0,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,10 percentile,4.99,3.3,precise,A
1,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,20 percentile,5.81,5.9,reasonably precise,A
2,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,25 percentile,6.17,6.5,reasonably precise,A
3,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,30 percentile,6.50,5.4,reasonably precise,A
4,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,40 percentile,7.21,7.8,reasonably precise,A
5,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,60 percentile,9.67,9.9,reasonably precise,A
6,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,70 percentile,11.09,11.0,use with caution,A
7,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,75 percentile,11.74,13.0,use with caution,A
8,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,80 percentile,12.34,16.0,use with caution,A
9,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,2002,Hourly,90 percentile,15.74,0.8,precise,A


## 3. Coverage, reliability & sanity checks

Coverage by year (units and ITL2 regions with a median), suppression by item, CV bands, then internal-consistency checks that should all pass: percentiles ordered P10 ≤ … ≤ P90, the same jobs count for weekly and hourly, implied full-time weekly hours (weekly ÷ hourly median) in a plausible ~35–45 range, and a list of large year-on-year jumps per unit (small-sample areas — treat with care). Year-on-year changes are only computed within a geography vintage.

In [5]:
med_w = ashe[(ashe.item == "Median") & (ashe.pay_type == "Weekly")]
cov = med_w.dropna(subset=["value"]).groupby("year").agg(units=("la_code","nunique"), itl2=("itl2_code","nunique"))
print("Units / ITL2 regions with a weekly median, by year (GB has 45 ITL2):")
print(cov.T.to_string())
gb_itl2 = set(spine.itl2_code) - {"TLN0"}
for y, g in med_w.dropna(subset=["value"]).groupby("year"):
    miss = gb_itl2 - set(g.itl2_code)
    if miss: print(f"  {y}: ITL2 with no LA median: {sorted(miss)}")
cur_cov = med_w.dropna(subset=["value"]).groupby("year").la_code_current.nunique()
print("\nCurrent LAs with a weekly median (itself or via predecessors), by year (GB = 350):")
print(cur_cov.to_frame().T.to_string())
for y, g in med_w.dropna(subset=["value"]).groupby("year"):
    miss = sorted(set(spine.la_code[~spine.la_code.str.startswith("N")]) - set(g.la_code_current))
    if miss: print(f"  {y}: current LAs with no median: {miss}")
print("Spine LAs never in ASHE:", sorted(set(spine.la_code) - set(ashe.la_code_current)), "(NI expected)")

print(f"\nSuppressed (no value): {ashe.value.isna().sum():,} of {len(ashe):,} cells. Status counts:")
print(ashe.status.value_counts().to_string())
share = (ashe[ashe.year.isin([2005, ashe.year.max()])].groupby(["item","year"]).value
             .apply(lambda s: s.isna().mean()).unstack().mul(100).round(0))
print("\n% of units suppressed (weekly+hourly pooled):"); print(share.to_string())
print("\nCV bands (non-missing cells):"); print(ashe.cv_band.value_counts().to_string())

# --- internal consistency ---
ORDER = ["10 percentile","20 percentile","25 percentile","30 percentile","40 percentile","Median",
         "60 percentile","70 percentile","75 percentile","80 percentile","90 percentile"]
w = ashe.pivot_table(index=["la_code","geo_vintage","year","pay_type"], columns="item", values="value")
bad = (w[ORDER].diff(axis=1) < 0).any(axis=1)
print(f"\nPercentile-order violations: {bad.sum()}")

jobs = w["Number of jobs"].unstack("pay_type")
print(f"Jobs weekly vs hourly, max abs difference: {(jobs.Weekly - jobs.Hourly).abs().max():.0f}")

med = w["Median"].unstack("pay_type")
hours = (med.Weekly / med.Hourly).dropna()
print(f"Implied FT weekly hours (weekly/hourly median): min {hours.min():.1f}, median {hours.median():.1f}, max {hours.max():.1f}")

yoy = (med.Weekly.unstack("year").pct_change(axis=1, fill_method=None).mul(100).stack())
print("\nWeekly median |YoY| > 25% (check before use):")
print(yoy[yoy.abs() > 25].round(1).to_string())

uk = med_w.groupby("year").value.median()
print("\nMedian across units of the weekly median (£) — watch 2004/2006/2011/2021 breaks:")
print(uk.round(0).to_frame().T.to_string())

Units / ITL2 regions with a weekly median, by year (GB has 45 ITL2):
year   2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
units   386   386   388   386   388   387   360   363   361   362   361   361   361   361   362   361   358   358   352   352   349   348   348   348
itl2     45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45

Current LAs with a weekly median (itself or via predecessors), by year (GB = 350):
year             2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
la_code_current   347   347   348   348   348   347   347   350   348   349   348   348   348   348   349   348   349   349   348   348   349   348   348   348
  2002: current LAs with no median: ['E06000053', 'E09000001', 'S12000023']
 